In [3]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

# Reload data fresh in this cell
train = pd.read_csv("../data/raw/train.csv")
X = train.drop(["Survived", "PassengerId", "Name", "Ticket", "Cabin"], axis=1)
y = train["Survived"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["str"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

# Build pipeline with XGBoost
xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(random_state=42, n_jobs=-1)),
])

# Define the hyperparameter search space
param_distributions = {
    "classifier__n_estimators":     [100, 200, 300, 500],
    "classifier__learning_rate":    [0.01, 0.05, 0.1, 0.2],
    "classifier__max_depth":        [3, 5, 7],
    "classifier__subsample":        [0.7, 0.85, 1.0],
    "classifier__colsample_bytree": [0.7, 0.85, 1.0],
}

# Run the search
print("Starting RandomizedSearchCV — this should take ~30-60 seconds...")
start = time.time()

search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=30,
    cv=5,
    scoring="roc_auc",        # optimise for AUC
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)

elapsed = time.time() - start
print(f"\nDone in {elapsed:.1f}s")
print(f"Best CV AUC: {search.best_score_:.4f}")
print(f"\nBest params:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

Starting RandomizedSearchCV — this should take ~30-60 seconds...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Done in 0.8s
Best CV AUC: 0.8617

Best params:
  classifier__subsample: 0.7
  classifier__n_estimators: 200
  classifier__max_depth: 3
  classifier__learning_rate: 0.05
  classifier__colsample_bytree: 0.7


In [4]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

best_model = search.best_estimator_

y_pred = best_model.predict(X_val)
y_proba = best_model.predict_proba(X_val)[:, 1]

print(f"Tuned XGBoost on validation set:")
print(f"Accuracy:  {accuracy_score(y_val, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, y_proba):.4f}")
print(f"Precision: {precision_score(y_val, y_pred):.4f}")
print(f"Recall:    {recall_score(y_val, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_val, y_pred):.4f}")
print()
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))
print()
print(classification_report(y_val, y_pred))

Tuned XGBoost on validation set:
Accuracy:  0.8156
ROC-AUC:   0.8829
Precision: 0.8154
Recall:    0.7162
F1 Score:  0.7626

Confusion Matrix:
[[93 12]
 [21 53]]

              precision    recall  f1-score   support

           0       0.82      0.89      0.85       105
           1       0.82      0.72      0.76        74

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.81       179
weighted avg       0.82      0.82      0.81       179

